In [1]:
import re
import pandas as pd
import os
from collections import defaultdict

def parse_dynamic_fed_log(log_file_path, output_csv_path=None):
    """
    解析具有动态延迟的联邦学习日志文件，提取每个epoch的信息并计算时间开销
    
    参数:
        log_file_path (str): 日志文件的路径
        output_csv_path (str, 可选): 输出CSV文件的路径
        
    返回:
        pandas.DataFrame: 包含解析结果的DataFrame
    """
    # 读取日志文件内容
    with open(log_file_path, 'r', encoding='utf-8') as file:
        log_lines = file.readlines()
    
    # 存储所有客户端最新的延迟信息
    client_delays = {}
    
    # 存储每个epoch的信息
    epochs_data = []
    
    # 当前处理的epoch和group
    current_epoch = None
    current_group = None
    group_clients = {}
    
    i = 0
    while i < len(log_lines):
        line = log_lines[i].strip()
        
        # 查找客户端上传和延迟信息
        upload_match = re.search(r'Client (\d+) uploaded at time: .*; simulated_delay: ([\d\.]+)', line)
        if upload_match:
            client_id = int(upload_match.group(1))
            delay = float(upload_match.group(2))
            client_delays[client_id] = delay
        
        # 查找group选择信息
        group_match = re.search(r'group (\d+) selected_clients: \[([\d, ]+)\]', line)
        if group_match:
            group_id = int(group_match.group(1))
            clients = [int(c.strip()) for c in group_match.group(2).split(',')]
            group_clients[group_id] = clients
        
        # 查找group_ready_num信息
        ready_match = re.search(r'group_ready_num: (\d+)', line)
        if ready_match:
            current_group = int(ready_match.group(1))
        
        # 查找epoch信息
        epoch_match = re.search(r'Epoch (\d+) tested, accuracy: ([\d\.]+) loss ([\d\.]+) run_time ([\d\.]+)', line)
        if epoch_match:
            current_epoch = int(epoch_match.group(1))
            accuracy = float(epoch_match.group(2))
            loss = float(epoch_match.group(3))
            run_time = float(epoch_match.group(4))
            
            # 确保当前group和clients信息存在
            if current_group is not None and current_group in group_clients:
                selected_clients = group_clients[current_group]
                
                # 计算这些客户端的最大延迟
                max_delay = 0.0
                for client_id in selected_clients:
                    if client_id in client_delays:
                        max_delay = max(max_delay, client_delays[client_id])
                
                # 添加epoch记录
                epochs_data.append({
                    'Epoch': current_epoch,
                    'Group_ID': current_group,
                    'Selected_Clients': selected_clients,
                    'Client_Count': len(selected_clients),
                    'Max_Delay': max_delay,
                    'Accuracy': accuracy,
                    'Loss': loss,
                    'Run_Time': run_time
                })
        
        i += 1
    
    # 创建DataFrame
    df = pd.DataFrame(epochs_data)
    
    # 如果提供了输出路径，保存到CSV
    if output_csv_path and not df.empty:
        # 将客户端列表转换为字符串格式以便保存
        df_to_save = df.copy()
        df_to_save['Selected_Clients'] = df_to_save['Selected_Clients'].apply(lambda x: ';'.join(map(str, x)))
        df_to_save.to_csv(output_csv_path, index=False)
        print(f"结果已保存到: {output_csv_path}")
    
    return df

def analyze_dynamic_delays(df):
    """
    分析动态延迟数据
    
    参数:
        df (pandas.DataFrame): 包含解析结果的DataFrame
        
    返回:
        dict: 分析统计结果
    """
    if df.empty:
        return {
            'total_epochs': 0,
            'avg_max_delay': 0,
            'min_delay': 0,
            'max_delay': 0,
            'avg_clients_per_group': 0,
            'avg_accuracy': 0,
            'final_accuracy': 0,
            'group_stats': {}
        }
    
    stats = {
        'total_epochs': len(df),
        'avg_max_delay': df['Max_Delay'].mean(),
        'min_delay': df['Max_Delay'].min(),
        'max_delay': df['Max_Delay'].max(),
        'avg_clients_per_group': df['Client_Count'].mean(),
        'avg_accuracy': df['Accuracy'].mean(),
        'final_accuracy': df.iloc[-1]['Accuracy'] if len(df) > 0 else 0,
        'group_stats': {}
    }
    
    # 按组统计
    for group_id in df['Group_ID'].unique():
        group_data = df[df['Group_ID'] == group_id]
        stats['group_stats'][group_id] = {
            'count': len(group_data),
            'avg_delay': group_data['Max_Delay'].mean(),
            'avg_accuracy': group_data['Accuracy'].mean()
        }
    
    return stats

def main(log_file_path, output_csv_path=None):
    """
    主函数，处理日志文件并输出结果
    
    参数:
        log_file_path (str): 日志文件的路径
        output_csv_path (str, 可选): 输出CSV文件的路径
    """
    print(f"正在解析日志文件: {log_file_path}")
    
    # 如果未提供输出路径，则创建默认路径
    if output_csv_path is None:
        dir_name = os.path.dirname(log_file_path)
        base_name = os.path.basename(log_file_path).split('.')[0]
        output_csv_path = os.path.join(dir_name, f"{base_name}_results.csv")
    
    # 解析日志文件
    df_results = parse_dynamic_fed_log(log_file_path, output_csv_path)
    
    # 显示前几行结果
    if not df_results.empty:
        print("\n解析结果预览:")
        print(df_results.head())
        
        # 分析统计
        stats = analyze_dynamic_delays(df_results)
        
        print("\n统计分析:")
        print(f"总共处理的epoch数: {stats['total_epochs']}")
        print(f"平均最大延迟: {stats['avg_max_delay']:.2f}")
        print(f"延迟范围: {stats['min_delay']:.2f} - {stats['max_delay']:.2f}")
        print(f"平均每组客户端数: {stats['avg_clients_per_group']:.2f}")
        print(f"平均准确率: {stats['avg_accuracy']:.2f}%")
        print(f"最终准确率: {stats['final_accuracy']:.2f}%")
        
        print("\n各组统计:")
        for group_id, group_stat in stats['group_stats'].items():
            print(f"Group {group_id}: {group_stat['count']} epochs, "
                f"平均延迟: {group_stat['avg_delay']:.2f}, "
                f"平均准确率: {group_stat['avg_accuracy']:.2f}%")
    else:
        print("未解析到任何数据。请检查日志文件格式是否正确。")

# 使用示例
if __name__ == "__main__":
    # 设置文件路径
    log_file_path = "output.log"
    output_csv_path = "federated_learning_results_dynamic.csv"
    
    # 运行主函数
    main(log_file_path, output_csv_path)

正在解析日志文件: output.log
结果已保存到: federated_learning_results_dynamic.csv

解析结果预览:
   Epoch  Group_ID                          Selected_Clients  Client_Count  \
0      1         0    [11, 1, 42, 34, 98, 10, 55, 6, 52, 77]            10   
1      2         1    [13, 57, 5, 4, 79, 14, 90, 37, 40, 71]            10   
2      3         2  [82, 24, 89, 97, 61, 36, 63, 39, 60, 84]            10   
3      4         3     [3, 27, 65, 47, 38, 9, 17, 54, 93, 8]            10   
4      5         4   [12, 75, 25, 73, 76, 43, 67, 35, 2, 53]            10   

   Max_Delay   Accuracy      Loss   Run_Time  
0       8.36  19.981971  2.270937  15.729416  
1       8.71  11.678686  2.298183  16.126892  
2       9.07   9.985978  2.293131  16.568580  
3       8.83  16.526442  2.309987  16.918038  
4       8.22  10.006010  2.299994  17.323868  

统计分析:
总共处理的epoch数: 2010
平均最大延迟: 8.34
延迟范围: 5.10 - 9.26
平均每组客户端数: 10.00
平均准确率: 71.49%
最终准确率: 77.55%

各组统计:
Group 0: 423 epochs, 平均延迟: 7.93, 平均准确率: 71.05%
Group 1: 382 epoch

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

def read_federated_learning_csv(csv_file_path):
    """
    读取联邦学习结果CSV文件
    
    参数:
        csv_file_path (str): CSV文件的路径
        
    返回:
        pandas.DataFrame: 包含联邦学习结果的DataFrame
    """
    try:
        # 读取CSV文件
        df = pd.read_csv(csv_file_path)
        
        # 处理Selected_Clients列，将其转换为列表
        if 'Selected_Clients' in df.columns:
            df['Selected_Clients'] = df['Selected_Clients'].apply(
                lambda x: [int(client) for client in x.split(';')] if isinstance(x, str) else x
            )
            
        # 添加一个列来表示每个epoch选择的客户端数量
        if 'Client_Count' not in df.columns and 'Selected_Clients' in df.columns:
            df['Client_Count'] = df['Selected_Clients'].apply(len)
            
        return df
    except Exception as e:
        print(f"读取CSV文件时出错: {str(e)}")
        return None

def analyze_results(df):
    """
    分析联邦学习结果
    
    参数:
        df (pandas.DataFrame): 联邦学习结果DataFrame
        
    返回:
        dict: 分析结果
    """
    if df is None or df.empty:
        return {
            'total_epochs': 0,
            'avg_accuracy': 0,
            'final_accuracy': 0,
            'avg_delay': 0,
            'group_stats': {}
        }
    
    analysis = {
        'total_epochs': len(df),
        'avg_accuracy': df['Accuracy'].mean(),
        'final_accuracy': df.iloc[-1]['Accuracy'],
        'accuracy_improvement': df.iloc[-1]['Accuracy'] - df.iloc[0]['Accuracy'],
        'avg_delay': df['Max_Delay'].mean() if 'Max_Delay' in df.columns else 0,
        'group_stats': {}
    }
    
    # 按组统计
    if 'Group_ID' in df.columns:
        for group_id in df['Group_ID'].unique():
            group_data = df[df['Group_ID'] == group_id]
            analysis['group_stats'][group_id] = {
                'count': len(group_data),
                'avg_accuracy': group_data['Accuracy'].mean(),
                'avg_delay': group_data['Max_Delay'].mean() if 'Max_Delay' in group_data.columns else 0,
                'client_selection_frequency': get_client_selection_frequency(group_data) if 'Selected_Clients' in group_data.columns else {}
            }
    
    return analysis

def get_client_selection_frequency(df):
    """
    统计客户端被选择的频率
    
    参数:
        df (pandas.DataFrame): 包含Selected_Clients列的DataFrame
        
    返回:
        dict: 客户端ID到选择频率的映射
    """
    if 'Selected_Clients' not in df.columns:
        return {}
    
    # 展平所有选择的客户端列表
    all_clients = []
    for clients in df['Selected_Clients']:
        all_clients.extend(clients)
    
    # 统计每个客户端出现的次数
    client_counts = {}
    for client in all_clients:
        if client in client_counts:
            client_counts[client] += 1
        else:
            client_counts[client] = 1
    
    # 计算频率
    total_selections = sum(client_counts.values())
    client_frequency = {client: count / total_selections for client, count in client_counts.items()}
    
    return client_frequency

def plot_accuracy_trend(df, output_path=None):
    """
    绘制准确率趋势图
    
    参数:
        df (pandas.DataFrame): 联邦学习结果DataFrame
        output_path (str, 可选): 输出图表的路径
    """
    if df is None or df.empty or 'Accuracy' not in df.columns or 'Epoch' not in df.columns:
        print("无法绘制准确率趋势图: 数据不完整")
        return
    
    plt.figure(figsize=(12, 6))
    
    # 将DataFrame列转换为NumPy数组以避免pandas的索引问题
    x = df['Epoch'].values
    y = df['Accuracy'].values
    
    plt.plot(x, y, marker='o', linestyle='-', color='blue')
    plt.title('联邦学习准确率趋势')
    plt.xlabel('Epoch')
    plt.ylabel('准确率 (%)')
    plt.grid(True)
    
    # 添加平均线
    avg_accuracy = df['Accuracy'].mean()
    plt.axhline(y=avg_accuracy, color='r', linestyle='--', label=f'平均准确率: {avg_accuracy:.2f}%')
    
    plt.legend()
    
    if output_path:
        plt.savefig(output_path)
        print(f"准确率趋势图已保存到: {output_path}")
    else:
        plt.show()
    
    plt.close()

def plot_group_delays(df, output_path=None):
    """
    绘制各组延迟对比图
    
    参数:
        df (pandas.DataFrame): 联邦学习结果DataFrame
        output_path (str, 可选): 输出图表的路径
    """
    if df is None or df.empty or 'Max_Delay' not in df.columns or 'Group_ID' not in df.columns:
        print("无法绘制组延迟图: 数据不完整")
        return
    
    # 按组计算平均延迟
    group_delays = df.groupby('Group_ID')['Max_Delay'].mean()
    
    plt.figure(figsize=(10, 6))
    
    # 将Series转换为NumPy数组以避免pandas的索引问题
    groups = group_delays.index.values
    delays = group_delays.values
    
    plt.bar(groups, delays, color='skyblue')
    plt.title('各组平均最大延迟对比')
    plt.xlabel('Group ID')
    plt.ylabel('平均最大延迟 (秒)')
    plt.grid(True, axis='y')
    plt.xticks(groups)
    
    # 添加数值标签
    for i, v in enumerate(delays):
        plt.text(groups[i], v + 0.1, f'{v:.2f}', ha='center')
    
    if output_path:
        plt.savefig(output_path)
        print(f"组延迟对比图已保存到: {output_path}")
    else:
        plt.show()
    
    plt.close()
    
def plot_max_delay_distribution(df, output_path=None):
    """
    绘制所有epoch的max_delay分布情况图
    
    参数:
        df (pandas.DataFrame): 联邦学习结果DataFrame
        output_path (str, 可选): 输出图表的路径
    """
    if df is None or df.empty or 'Max_Delay' not in df.columns:
        print("无法绘制max_delay分布图: 数据不完整")
        return
    
    # 设置字体
    config = {
        "font.family": 'serif',
        "font.size": 12,
        "mathtext.fontset": 'stix',
        "font.serif": ['SimSun'],
    }
    plt.rcParams.update(config)
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['axes.grid'] = True
    
    # 创建一个包含2个子图的图表
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
    
    # 子图1: 直方图+核密度估计
    delay_values = df['Max_Delay'].values
    
    # 使用Freedman-Diaconis规则计算bin宽度
    q75, q25 = np.percentile(delay_values, [75, 25])
    iqr = q75 - q25
    bin_width = 2 * iqr / (len(delay_values) ** (1/3))
    num_bins = int(np.ceil((max(delay_values) - min(delay_values)) / bin_width))
    
    # 至少10个bin，最多50个bin
    num_bins = max(10, min(num_bins, 50))
    
    # 绘制直方图
    n, bins, patches = ax1.hist(delay_values, bins=num_bins, density=True, alpha=0.7, color='skyblue', edgecolor='black')
    
    # 添加核密度估计
    try:
        from scipy.stats import gaussian_kde
        kde = gaussian_kde(delay_values)
        x = np.linspace(min(delay_values), max(delay_values), 1000)
        ax1.plot(x, kde(x), 'r-', linewidth=2)
        ax1.set_xlabel('Max_Delay (秒)')
        ax1.set_ylabel('密度')
        ax1.set_title('Max_Delay分布直方图与核密度估计')
        ax1.grid(True, linestyle='--', alpha=0.7)
    except Exception as e:
        print(f"绘制核密度估计时出错: {str(e)}")
    
    # 子图2: 箱型图
    # 先按照Group_ID创建箱型图
    if 'Group_ID' in df.columns:
        ax2.boxplot([df[df['Group_ID'] == group]['Max_Delay'].values for group in sorted(df['Group_ID'].unique())],
                  labels=sorted(df['Group_ID'].unique()),
                  patch_artist=True)
        
        # 设置箱型图颜色
        colors = ['lightblue', 'lightgreen', 'lightcoral', 'lightyellow', 'lightpink']
        for patch, color in zip(ax2.get_children()[4:4+len(df['Group_ID'].unique())], 
                             colors * (len(df['Group_ID'].unique()) // len(colors) + 1)):
            if hasattr(patch, 'set_facecolor'):
                patch.set_facecolor(color)
    else:
        # 如果没有Group_ID，就只画一个总体的箱型图
        ax2.boxplot(delay_values, patch_artist=True)
        ax2.get_children()[4].set_facecolor('lightblue')
    
    ax2.set_xlabel('Group ID')
    ax2.set_ylabel('Max_Delay (秒)')
    ax2.set_title('Max_Delay箱型图')
    ax2.grid(True, linestyle='--', alpha=0.7)
    
    # 添加全局标题
    fig.suptitle('所有Epoch的Max_Delay分布情况', fontsize=16)
    
    # 添加统计信息
    stats_text = (f"统计信息:\n"
                 f"样本数: {len(delay_values)}\n"
                 f"均值: {np.mean(delay_values):.2f}\n"
                 f"中位数: {np.median(delay_values):.2f}\n"
                 f"标准差: {np.std(delay_values):.2f}\n"
                 f"最小值: {np.min(delay_values):.2f}\n"
                 f"最大值: {np.max(delay_values):.2f}\n"
                 f"25%分位数: {np.percentile(delay_values, 25):.2f}\n"
                 f"75%分位数: {np.percentile(delay_values, 75):.2f}")
    
    fig.text(0.02, 0.02, stats_text, fontsize=10, 
             bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.5'))
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"Max_Delay分布图已保存到: {output_path}")
    else:
        plt.show()
    
    plt.close()

def plot_accuracy_by_group(df, output_path=None):
    """
    绘制各组准确率对比图
    
    参数:
        df (pandas.DataFrame): 联邦学习结果DataFrame
        output_path (str, 可选): 输出图表的路径
    """
    if df is None or df.empty or 'Accuracy' not in df.columns or 'Group_ID' not in df.columns:
        print("无法绘制组准确率图: 数据不完整")
        return
    
    # 按组计算平均准确率
    group_accuracy = df.groupby('Group_ID')['Accuracy'].mean()
    
    plt.figure(figsize=(10, 6))
    
    # 将Series转换为NumPy数组以避免pandas的索引问题
    groups = group_accuracy.index.values
    accuracies = group_accuracy.values
    
    plt.bar(groups, accuracies, color='lightgreen')
    plt.title('各组平均准确率对比')
    plt.xlabel('Group ID')
    plt.ylabel('平均准确率 (%)')
    plt.grid(True, axis='y')
    plt.xticks(groups)
    
    # 添加数值标签
    for i, v in enumerate(accuracies):
        plt.text(groups[i], v + 0.5, f'{v:.2f}%', ha='center')
    
    if output_path:
        plt.savefig(output_path)
        print(f"组准确率对比图已保存到: {output_path}")
    else:
        plt.show()
    
    plt.close()

def generate_report(df, output_dir=None):
    """
    生成分析报告
    
    参数:
        df (pandas.DataFrame): 联邦学习结果DataFrame
        output_dir (str, 可选): 输出目录
    """
    if df is None or df.empty:
        print("无法生成报告: 数据为空")
        return
    
    # 创建输出目录
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    # 进行分析
    analysis = analyze_results(df)
    
    # 生成报告文本
    report = "# 联邦学习分析报告\n\n"
    report += f"## 基础统计\n"
    report += f"- 总epoch数: {analysis['total_epochs']}\n"
    report += f"- 平均准确率: {analysis['avg_accuracy']:.2f}%\n"
    report += f"- 最终准确率: {analysis['final_accuracy']:.2f}%\n"
    report += f"- 准确率提升: {analysis['accuracy_improvement']:.2f}%\n"
    report += f"- 平均最大延迟: {analysis['avg_delay']:.2f}秒\n\n"
    
    report += f"## 各组统计\n"
    for group_id, stats in analysis['group_stats'].items():
        report += f"### Group {group_id}\n"
        report += f"- 参与epoch次数: {stats['count']}\n"
        report += f"- 平均准确率: {stats['avg_accuracy']:.2f}%\n"
        report += f"- 平均最大延迟: {stats['avg_delay']:.2f}秒\n"
        
        # 添加客户端选择频率统计
        if stats['client_selection_frequency']:
            report += f"- 客户端选择频率 (前5):\n"
            sorted_clients = sorted(stats['client_selection_frequency'].items(), 
                                   key=lambda x: x[1], reverse=True)[:5]
            for client, freq in sorted_clients:
                report += f"  - 客户端 {client}: {freq*100:.2f}%\n"
        
        report += "\n"
    
    # 保存报告
    if output_dir:
        report_path = os.path.join(output_dir, "federated_learning_report.md")
        with open(report_path, 'w') as f:
            f.write(report)
        print(f"分析报告已保存到: {report_path}")
    else:
        print(report)
    
    # 生成图表
    if output_dir:
        try:
            plot_accuracy_trend(df, os.path.join(output_dir, "accuracy_trend.png"))
        except Exception as e:
            print(f"生成准确率趋势图时出错: {str(e)}")
        
        try:
            plot_group_delays(df, os.path.join(output_dir, "group_delays.png"))
        except Exception as e:
            print(f"生成组延迟对比图时出错: {str(e)}")
            
        try:
            plot_max_delay_distribution(df, os.path.join(output_dir, "max_delay_distribution.png"))
        except Exception as e:
            print(f"生成max_delay分布图时出错: {str(e)}")
            
        try:
            plot_accuracy_by_group(df, os.path.join(output_dir, "group_accuracy.png"))
        except Exception as e:
            print(f"生成组准确率对比图时出错: {str(e)}")
    else:
        try:
            plot_accuracy_trend(df)
            plot_group_delays(df)
            plot_max_delay_distribution(df)
            plot_accuracy_by_group(df)
        except Exception as e:
            print(f"生成图表时出错: {str(e)}")

def main(csv_file_path, output_dir=None):
    """
    主函数
    
    参数:
        csv_file_path (str): CSV文件的路径
        output_dir (str, 可选): 输出目录
    """
    print(f"正在读取CSV文件: {csv_file_path}")
    df = read_federated_learning_csv(csv_file_path)
    
    if df is not None and not df.empty:
        print(f"成功读取数据: {len(df)}行")
        print("\n数据预览:")
        print(df.head())
        
        # 生成报告
        try:
            generate_report(df, output_dir)
        except Exception as e:
            print(f"生成报告时出错: {str(e)}")
    else:
        print("无法读取数据，请检查文件路径和格式")

if __name__ == "__main__":
    # 设置文件路径
    csv_file_path = "federated_learning_results_dynamic.csv"
    output_dir = "federated_learning_analysis"
    
    # 运行主函数
    main(csv_file_path, output_dir)

正在读取CSV文件: federated_learning_results_dynamic.csv
成功读取数据: 2010行

数据预览:
   Epoch  Group_ID                          Selected_Clients  Client_Count  \
0      1         0    [11, 1, 42, 34, 98, 10, 55, 6, 52, 77]            10   
1      2         1    [13, 57, 5, 4, 79, 14, 90, 37, 40, 71]            10   
2      3         2  [82, 24, 89, 97, 61, 36, 63, 39, 60, 84]            10   
3      4         3     [3, 27, 65, 47, 38, 9, 17, 54, 93, 8]            10   
4      5         4   [12, 75, 25, 73, 76, 43, 67, 35, 2, 53]            10   

   Max_Delay   Accuracy      Loss   Run_Time  
0       8.36  19.981971  2.270937  15.729416  
1       8.71  11.678686  2.298183  16.126892  
2       9.07   9.985978  2.293131  16.568580  
3       8.83  16.526442  2.309987  16.918038  
4       8.22  10.006010  2.299994  17.323868  
分析报告已保存到: federated_learning_analysis/federated_learning_report.md


/tmp/ipykernel_65902/292730613.py:139: UserWarning: Glyph 32852 (\N{CJK UNIFIED IDEOGRAPH-8054}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:139: UserWarning: Glyph 37030 (\N{CJK UNIFIED IDEOGRAPH-90A6}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:139: UserWarning: Glyph 23398 (\N{CJK UNIFIED IDEOGRAPH-5B66}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:139: UserWarning: Glyph 20064 (\N{CJK UNIFIED IDEOGRAPH-4E60}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:139: UserWarning: Glyph 20934 (\N{CJK UNIFIED IDEOGRAPH-51C6}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:139: UserWarning: Glyph 30830 (\N{CJK UNIFIED IDEOGRAPH-786E}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:139: UserWarning: Glyph 29575 (\N{CJK UNIFIED IDEOGRAPH-7387

准确率趋势图已保存到: federated_learning_analysis/accuracy_trend.png
组延迟对比图已保存到: federated_learning_analysis/group_delays.png


/tmp/ipykernel_65902/292730613.py:179: UserWarning: Glyph 21508 (\N{CJK UNIFIED IDEOGRAPH-5404}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:179: UserWarning: Glyph 32452 (\N{CJK UNIFIED IDEOGRAPH-7EC4}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:179: UserWarning: Glyph 24179 (\N{CJK UNIFIED IDEOGRAPH-5E73}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:179: UserWarning: Glyph 22343 (\N{CJK UNIFIED IDEOGRAPH-5747}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:179: UserWarning: Glyph 26368 (\N{CJK UNIFIED IDEOGRAPH-6700}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:179: UserWarning: Glyph 22823 (\N{CJK UNIFIED IDEOGRAPH-5927}) missing from current font.
  plt.savefig(output_path)
/tmp/ipykernel_65902/292730613.py:179: UserWarning: Glyph 24310 (\N{CJK UNIFIED IDEOGRAPH-5EF6

Max_Delay分布图已保存到: federated_learning_analysis/max_delay_distribution.png
组准确率对比图已保存到: federated_learning_analysis/group_accuracy.png
